# Test Results and Visualizations

## Overview
This notebook evaluates the Machine Learning models trained in Notebook 02 using **subject‑wise out‑of‑fold predictions**. This approach guarantees that no subject appears in both training and test sets, providing an honest estimate of generalization performance.

We will:
- Load the feature matrix and the saved models.
- Generate out‑of‑fold predictions via 5‑fold `GroupKFold` (matching Notebook 02).
- Compute comprehensive classification metrics.
- Visualize results with confusion matrices, ROC‑AUC curves, and comparative bar charts.
- Cross‑check with the cross‑validation scores saved from Notebook 02.

## 1. Setup and Data Loading
We begin by importing necessary libraries and loading the feature matrix, subject identifiers, and label encoder saved from Notebook 01/02

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os
import warnings

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score, f1_score, 
    precision_score, recall_score
)
from sklearn.multiclass import OneVsRestClassifier
from itertools import cycle

PROCESSED_DIR = '../data/processed/'
MODELS_DIR    = '../data/processed/models/'

warnings.filterwarnings('ignore', category=UserWarning)  # clean output

sns.set_theme(style='whitegrid', palette='muted')
print('All imports successful.')

In [ ]:
# Load feature matrix
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'eeg_rbp_features.csv'))

# Load metadata saved by Notebook 02
le           = joblib.load(os.path.join(MODELS_DIR, 'label_encoder.joblib'))
FEATURE_COLS = joblib.load(os.path.join(MODELS_DIR, 'feature_cols.joblib'))

METADATA_COLS = ['subject_id', 'epoch_id', 'label']
X      = df[FEATURE_COLS].values
groups = df['subject_id'].values
y      = le.transform(df['label'].values)

CLASS_NAMES = list(le.classes_)   # e.g. ['AD', 'CN', 'FTD']
N_CLASSES   = len(CLASS_NAMES)

print(f'Dataset shape : {X.shape}')
print(f'Classes       : {CLASS_NAMES}')
print(f'Unique subjects: {len(np.unique(groups))}')

## 2. Load Trained Models
The final models (with optimal hyperparameters selected via nested cross‑validation) were saved in Notebook 02.  
We load them here to generate out‑of‑fold predictions without re‑tuning.

In [ ]:
# Load all trained models from Notebook 02
MODEL_FILES = {
    'KNN':                 'knn.joblib',
    'Decision Tree':       'decision_tree.joblib',
    'SVM':                 'svm.joblib',
    'Logistic Regression': 'logistic_regression.joblib',
    'LDA':                 'lda.joblib',
}

MODELS = {}
for name, fname in MODEL_FILES.items():
    path = os.path.join(MODELS_DIR, fname)
    MODELS[name] = joblib.load(path)
    print(f'  Loaded: {name}')

print(f'\n✓ {len(MODELS)} models loaded.')

### Why Are We Re‑running GroupKFold in This Notebook?

You might wonder: *We already trained and evaluated the models in Notebook 02. Why are we fitting them again here?*

The answer lies in **what we need to produce**.

| Notebook 02 (Nested CV) | Notebook 03 (Out‑of‑Fold Predictions) |
| :--- | :--- |
| Provides **aggregate performance estimates** (mean accuracy, mean F1‑score ± standard deviation). | Needs **individual predictions for every single epoch** to construct confusion matrices and ROC curves. |
| Saves a **final model** trained on the entire dataset (with optimal hyperparameters). | Cannot use that final model directly on the same data, because the model has already "seen" those epochs during training – this would yield **overly optimistic, biased metrics**. |

#### The Solution: Out‑of‑Fold (OOF) Predictions

**Out‑of‑Fold (OOF) predictions** are model predictions made on data that was **not used during the training of that specific model**.

In the context of K‑Fold cross‑validation:
1. The dataset is split into *K* equally sized folds.
2. For each fold *i*:
   - A model is trained on the other *K‑1* folds (the **training set**).
   - Predictions are generated for the **held‑out fold** *i* (the **validation set**).

#### Applying OOF
We **re‑run a 5‑fold `GroupKFold` split** (identical to the outer loop of Notebook 02) but **without any hyperparameter tuning**. In each fold:

1. We train the pipeline (using the **fixed, optimal hyperparameters** found in Notebook 02) on a subset of subjects.
2. We predict the held‑out subjects that were **completely unseen** during that fold's training.

After looping through all folds, every epoch has a prediction that was made by a model that **never saw that subject in its training set**. These *out‑of‑fold* predictions are **unbiased** and can be safely used to compute per‑class metrics, confusion matrices, and ROC curves.

#### Isn't This What Notebook 02 Already Did?

Not exactly. Notebook 02 computed **metrics per fold** and then **averaged** them. It did **not** store the individual predictions needed for the visualizations we create here. By generating OOF predictions now, we obtain the detailed, epoch‑level results required for clinical interpretability, while preserving the strict subject‑wise separation that prevents data leakage.

In [ ]:
# Generate subject-wise test predictions using GroupKFold
# This mirrors the exact split used in Notebook 02 to maintain integrity
gkf = GroupKFold(n_splits=5)

# Collect out-of-fold predictions for each model
oof_preds  = {name: np.zeros(len(y), dtype=int)         for name in MODELS}
oof_probas = {name: np.zeros((len(y), N_CLASSES))       for name in MODELS}

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train         = y[train_idx]

    for name, pipeline in MODELS.items():
        # We retrain the pipeline from scratch on the training subjects.
        # This mimics the outer loop of nested CV and yields unbiased predictions for the test subjects.
        pipeline.fit(X_train, y_train)
        oof_preds[name][test_idx]  = pipeline.predict(X_test)
        oof_probas[name][test_idx] = pipeline.predict_proba(X_test)

    print(f'  Fold {fold+1}/5 complete')

print('\n✓ Out-of-fold predictions generated for all models.')

## 3. Quantitative Performance Metrics
We compute standard classification metrics for each model: accuracy, macro‑averaged precision, recall, and F1‑score.  
Per‑class metrics are shown in the detailed classification reports.

In [ ]:
# Comprehensive classification report for each model (precision, recall, f1-score per class)
for name in MODELS:
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(classification_report(y, oof_preds[name], target_names=CLASS_NAMES))

In [ ]:
# Build a comparative summary DataFrame

rows = []
for name in MODELS:
    rows.append({
        'Model'    : name,
        'Accuracy' : accuracy_score(y, oof_preds[name]),
        'Precision': precision_score(y, oof_preds[name], average='macro', zero_division=0),
        'Recall'   : recall_score(y, oof_preds[name], average='macro', zero_division=0),
        'F1 Macro' : f1_score(y, oof_preds[name], average='macro', zero_division=0),
    })

summary_df = pd.DataFrame(rows).set_index('Model').sort_values('F1 Macro', ascending=False)

print('=== Comparative Summary Table ===')
display(summary_df.round(4))

best = summary_df['F1 Macro'].idxmax()
print(f'\n🏆 Best model by F1-Macro: {best} ({summary_df.loc[best, "F1 Macro"]:.4f})')

## 4. Confusion Matrices
Confusion matrices show how predictions are distributed across the three diagnostic classes (AD, CN, FTD).  
Cells are normalized by true class and annotated with absolute counts.  
This helps identify systematic misclassifications (e.g., AD vs. FTD).

In [ ]:
# Build Confusion Matrices for each model
n_models = len(MODELS)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))

for ax, name in zip(axes, MODELS):
    cm = confusion_matrix(y, oof_preds[name])
    
    # Normalize per true class (row) for easier reading
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(
        cm_norm, annot=cm, fmt='d',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', vmin=0, vmax=1,
        linewidths=0.5, linecolor='gray',
        ax=ax, cbar=False
    )
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

fig.suptitle('Confusion Matrices — Normalized (counts annotated)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

### Interpreting the Raw Prediction Counts Table

The table above shows the **absolute number of out‑of‑fold predictions** made by each model across all 5 folds of the subject‑wise cross‑validation.

- **Rows** correspond to the **true diagnostic label** (`AD`, `CN`, or `FTD`).
- **Columns** correspond to the **predicted label**.
- **Cell values** represent how many EEG epochs with a given true label were classified as a specific predicted label.

For example, a value of `7828` in the `KNN` column under `AD` → `AD` means that KNN correctly identified 7,828 epochs belonging to Alzheimer’s disease patients. The other numbers in the same column (e.g., `AD` → `CN` = 3936) indicate misclassifications.

These counts are the raw data used to compute the confusion matrices and classification metrics (precision, recall, F1‑score) shown later in the notebook. They allow you to quickly spot systematic errors, such as whether the models tend to confuse Alzheimer’s disease with Frontotemporal dementia.

## 5. ROC‑AUC Curves
Receiver Operating Characteristic (ROC) curves and the corresponding Area Under the Curve (AUC) are plotted for each class using a one‑vs‑rest strategy.  
AUC values above 0.5 indicate discriminative power beyond random guessing.

In [ ]:
# Binarize labels for One-vs-Rest ROC
y_bin = label_binarize(y, classes=list(range(N_CLASSES)))

colors = cycle(['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#264653'])
line_styles = ['-', '--', '-.', ':', '-']

fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5), sharey=True)

for ax, (name, color) in zip(axes, zip(MODELS, colors)):
    probas = oof_probas[name]
    
    auc_scores = []
    for i, (cls_name, ls) in enumerate(zip(CLASS_NAMES, line_styles)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probas[:, i])
        roc_auc     = auc(fpr, tpr)
        auc_scores.append(roc_auc)
        ax.plot(fpr, tpr, lw=2, linestyle=ls,
                label=f'{cls_name} (AUC={roc_auc:.3f})')
    
    # Macro-average AUC
    mean_auc = np.mean(auc_scores)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
    ax.set_title(f'{name}\nMacro AUC={mean_auc:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('True Positive Rate')
fig.suptitle('ROC-AUC Curves — One-vs-Rest Multi-class',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'roc_auc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_auc_curves.png')

## 6. Comparative Summary and Visualization
A side‑by‑side bar chart compares the macro‑averaged metrics across all models.  
The dashed line represents the random‑guess baseline (33.3% for 3 classes).

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Macro']
x       = np.arange(len(summary_df))
width   = 0.2
palette = ['#457b9d', '#2a9d8f', '#e9c46a', '#e63946']

fig, ax = plt.subplots(figsize=(12, 5))

for i, (metric, color) in enumerate(zip(metrics, palette)):
    bars = ax.bar(x + i * width, summary_df[metric], width,
                  label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(summary_df.index, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.axhline(y=1/N_CLASSES, color='gray', linestyle='--', lw=1,
           label=f'Random baseline ({1/N_CLASSES:.2f})')
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'performance_comparison.png'), dpi=150)
plt.show()
print('Saved: performance_comparison.png')

## 7. Per‑Class F1 Heatmap
This heatmap breaks down the F1‑score for each individual class (AD, CN, FTD) and model.  
It reveals which classes are easier/harder to classify and whether certain models excel at specific diagnoses.

In [ ]:
# Build per-class F1 matrix (models × classes)
f1_matrix = []
for name in MODELS:
    report = classification_report(y, oof_preds[name],
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    f1_matrix.append([report[cls]['f1-score'] for cls in CLASS_NAMES])

f1_df = pd.DataFrame(f1_matrix, index=list(MODELS.keys()), columns=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(f1_df, annot=True, fmt='.3f', cmap='YlGnBu',
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Per-Class F1-Score by Model', fontsize=13, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'f1_per_class_heatmap.png'), dpi=150)
plt.show()
print('Saved: f1_per_class_heatmap.png')

>The heatmap reveals that **Frontotemporal Dementia (FTD) is the most challenging class to classify**, with F1-scores ranging from ~0.12 to 0.25 across models. This is not a flaw in the pipeline, but rather a reflection of **neurophysiological and dataset-specific factors**:


## 8. Cross‑Check with Nested CV Results
The out‑of‑fold metrics obtained here should closely match the nested cross‑validation scores saved from Notebook 02.  
We load the `cv_summary.csv` file and compare the two sets of estimates to ensure consistency.

In [ ]:
# Load the CV summary saved by Notebook 02
cv_summary_path = os.path.join(PROCESSED_DIR, 'cv_summary.csv')
if os.path.exists(cv_summary_path):
    cv_df = pd.read_csv(cv_summary_path, index_col=0)
    print('=== Cross-Validation Summary from Notebook 02 ===')
    display(cv_df.round(4))
    
    # Merge with OOF summary for direct comparison
    comp = summary_df[['Accuracy', 'F1 Macro']].join(
        cv_df[['Accuracy', 'F1 Macro']], lsuffix='_OOF', rsuffix='_CV'
    )
    print('\n=== Comparison: Out-of-Fold vs Nested CV ===')
    display(comp.round(4))
else:
    print('CV summary file not found. Run Notebook 02 first.')

## 9. Final Summary
The best performing model (according to macro F1‑score) is identified, and its key metrics are reported below.  
All generated figures are saved in the `processed/` directory.

In [ ]:
best_model  = summary_df['F1 Macro'].idxmax()
best_f1     = summary_df.loc[best_model, 'F1 Macro']
best_acc    = summary_df.loc[best_model, 'Accuracy']

print('=' * 55)
print('  FINAL RESULTS SUMMARY (out-of-fold predictions)')
print('=' * 55)
print(f'  Best model    : {best_model}')
print(f'  Accuracy      : {best_acc:.4f}')
print(f'  F1 Macro      : {best_f1:.4f}')
print('=' * 55)
print(f'\nFiles saved to: {PROCESSED_DIR}')
print('  - confusion_matrices.png')
print('  - roc_auc_curves.png')
print('  - performance_comparison.png')
print('  - f1_per_class_heatmap.png')
print('\n✓ Analysis complete.')